# The same ranking function, two front-ends

The counter below is modelled twice — as a **gym environment** (`GymCounter`) and as a
hand-written **`sugar.Module`** (`SugarCounter`) — over one shared set of state wires. We then
show two things:

1. the two encodings define the **identical transition relation** (proved with Z3), and
2. a single trained neural **ranking function** verifies **identically** against both.

The program: `x` increments while `x < y` or `x < z`, and resets to `0` otherwise; `y` and `z`
are constant. `R(x, y, z)` is a *liveness* certificate — while not at a target (`x == y` or
`x == z`) it stays non-negative and strictly decreases, so a target is always reached.

In [1]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
import z3

from zrth import LRA, Var, Real, X
from zrth import sugar
from zrth.sugar import ite
from zrth.gym import Env
from zrth.torch import Module
from zrth import z3 as zz3

## Shared state variables

Both front-ends are built from the **same** variables (`Var`) — each stands for its latched
wire, with `X(v)` its next value — so the comparison is over identical wires (and the state
ordering can't drift between the two).

In [2]:
REAL = Real([1, 1])

x, y, z = Var(REAL), Var(REAL), Var(REAL)
state_vars = [x, y, z]

## The gym front-end

The private-state sorts are declared explicitly via `attrs` (there is no inference) — so there
are **no dummy `self.x/y/z = …` assignments in `__init__`** whose only job would be to let the
analyzer guess the sorts.

In [3]:
class GymCounter(gym.Env):
    "x increments while x < y or x < z, resets otherwise; y and z stay constant."

    def __init__(self, y0, z0):
        super().__init__()
        self.action_space = spaces.Discrete(1)
        self.observation_space = spaces.Box(low=-1e6, high=1e6, shape=(1,))
        self.y0, self.z0 = y0, z0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.x = 0.0
        self.y = self.y0
        self.z = self.z0
        return self.x, {}

    def step(self, action):
        if self.x < self.y or self.x < self.z:
            self.x = self.x + 1.0
        else:
            self.x = 0.0
        self.y = self.y
        self.z = self.z
        at_target = self.x == self.y or self.x == self.z
        return self.x, (1.0 if at_target else 0.0), at_target, False, {}


gym_module = Env(GymCounter(y0=30.0, z0=50.0),
              attrs={"x": x, "y": y, "z": z})
print(gym_module)

<Env<GymCounter instance>>


## The sugar front-end

The same transition, written directly: `x' = ite(x < y or x < z, x + 1, 0)`, with `y`, `z`
held. The `ctrl` variables arrive as unpacked parameters of `update`.

In [4]:
class SugarCounter(sugar.Module):
    def init(self):
        return 0.0, 0.0, 0.0                      # initial state (not used by the obligation)

    def update(self, x, y, z):
        loop = (x < y) | (x < z)
        return ite(loop, x + 1.0, 0.0), y, z


sugar_module = SugarCounter(theory=LRA, ctrl=(x, y, z))
print(sugar_module.with_varnames({x : "x", y : "y", z : "z"}))

module
  interface
    x : Real([1,1])
    y : Real([1,1])
    z : Real([1,1])
  atom controls x, y, z reads x, y, z
  init
    #45 := [[0]] 
    X(x) := Id #45
    #46 := [[0]] 
    X(y) := Id #46
    #47 := [[0]] 
    X(z) := Id #47
  delay
    d(x) := ZERO 
    d(y) := ZERO 
    d(z) := ZERO 
  update
    #48 := Lt (x, y)
    #49 := Lt (x, z)
    #50 := Or (#48, #49)
    #51 := [[1]] 
    #52 := Add (x, #51)
    #53 := [[0]] 
    #54 := Ite (#50, #52, #53)
    X(x) := Id #54
    X(y) := Id y
    X(z) := Id z



## 1. The two encodings define the same transition

Run each module's `update` once from the same symbolic latched state and check the next-state
expressions are equivalent — no ranking function involved. (The gym `or` compiles to a nested
`If` and sugar's to `Or`; Z3 proves them equal.)

In [5]:
import numpy as np
def next_state(program, seed):
    "Run `program`'s update once from a seeded latched state; return the wire -> value map."
    s = {var: v for var, v in zip(state_vars, seed)}
    for atom in program.atoms:
        for term in atom.update:
            s.update(zip(term.write, zz3.eval(term.itype, [s[w] for w in term.read])))
    return s


fx, fy, fz = z3.Real("x"), z3.Real("y"), z3.Real("z")
after_gym   = next_state(gym_module,   [np.array([fx]), np.array([fy]), np.array([fz])])
after_sugar = next_state(sugar_module, [np.array([fx]), np.array([fy]), np.array([fz])])

for v, name in zip(state_vars, ("x", "y", "z")):
    eg, es = after_gym[X(v)].item(), after_sugar[X(v)].item()
    solver = z3.Solver(); solver.add(eg != es)
    equivalent = solver.check() == z3.unsat
    print(f"{name}':   gym = {eg}")
    print(f"       sugar = {es}    equivalent: {equivalent}")
    assert equivalent

x':   gym = If(If(x < y, True, x < z), x + 1, 0)
       sugar = If(Or(x < y, x < z), x + 1, 0)    equivalent: True
y':   gym = y
       sugar = y    equivalent: True
z':   gym = z
       sugar = z    equivalent: True


## The ranking function

`R(x, y, z)` is a small ReLU network. We train it on rollouts of the gym counter so that, away
from a target, `R >= 0` and `R` decreases by at least `margin` each step.

In [6]:
class RankingNN(nn.Module):
    "R(x, y, z) -> scalar;  [3] -> 2 -> [1]."

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 2)
        self.fc2 = nn.Linear(2, 1)

    def forward(self, s):
        return self.fc2(torch.relu(self.fc1(s)))


torch.manual_seed(2)
ranking_module = Module(RankingNN())

opt = torch.optim.Adam(ranking_module.parameters(), lr=0.01)
margin = 0.1
env = GymCounter(30.0, 50.0)
for epoch in range(201):
    env.reset()
    traj = []
    for _ in range(40):
        traj.append((env.x, env.y, env.z))
        env.step(0)
    loss = torch.tensor(0.0)
    for (sx, sy, sz), nxt in zip(traj, traj[1:]):
        if sx == sy or sx == sz:                  # at a target -> no obligation
            continue
        r  = ranking_module(torch.tensor((sx, sy, sz)).float().unsqueeze(0)).squeeze()
        rn = ranking_module(torch.tensor(nxt).float().unsqueeze(0)).squeeze()
        loss = loss + torch.relu(rn - r + margin) + torch.relu(-r)
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  loss = {loss.item():.4f}")

epoch   0  loss = 278.9917
epoch  40  loss = 6.9440
epoch  80  loss = 5.4820
epoch 120  loss = 3.9356
epoch 160  loss = 2.2787
epoch 200  loss = 0.0000


## 2. The same ranking function verifies against both

`obligation` is front-end agnostic: it symbolically runs a module's `update`, evaluates the
**same** `ranking_module` on the current and next state, and checks `R >= 0` and that `R`
decreases by `margin/2` on the in-loop domain (`y == 30`, `z == 50`, `x >= 0` and still looping).

In [27]:
def obligation(program, ranking, margin, domain):
    "(R >= 0, R decreases by margin/2) on `domain`, for `program`'s transition."
    s = {v: np.array([z3.FreshReal()]) for v in state_vars}
    for atom in program.atoms:
        for term in atom.update:
            s.update(zip(term.write, zz3.eval(term.itype, [s[w] for w in term.read])))
    cur  = [s[v].item() for v in state_vars]
    nxt_ = [s[X(v)].item() for v in state_vars]

    ext, itf = list(ranking.extl), list(ranking.intf)
    def R(vec):
        r = {X(ext[0]): np.array(vec)}
        for atom in ranking.atoms:
            for term in atom.update:
                r.update(zip(term.write, zz3.eval(term.itype, [r[w] for w in term.read])))
        return r[X(itf[0])]

    dom = domain(cur)
    def holds(negation):
        solver = z3.Solver(); solver.add(*dom); solver.add(negation)
        return solver.check() == z3.unsat

    return holds(R(cur)[0][0] < 0), holds(R(nxt_)[0][0] >= R(cur)[0][0] - margin / 2)


domain = lambda cur: [cur[1] == 30, cur[2] == 50,
                      z3.And(cur[0] >= 0, z3.Or(cur[0] < cur[1], cur[0] < cur[2]))]

res_gym   = obligation(gym_module,   ranking_module, margin, domain)
res_sugar = obligation(sugar_module, ranking_module, margin, domain)
print(f"gym_module    (R >= 0, decreases): {res_gym}")
print(f"sugar_module  (R >= 0, decreases): {res_sugar}")
assert res_gym == res_sugar == (True, True)
print("\nThe same ranking function verifies identically for both front-ends.")

gym_module    (R >= 0, decreases): (False, False)
sugar_module  (R >= 0, decreases): (False, False)


AssertionError: 

## Result

The gym-derived module and the hand-written `sugar.Module` have the same transition relation, and
one trained ranking function certifies the liveness property for both — the ranking function is
front-end agnostic.

Note: because the two modules share wires, they are meant to be verified **separately**; composing
them (`Module.comp(gym_module, sugar_module)`) would make both drive the same `X(x)` wire and be
rejected.